In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import cv2
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, jaccard_score, precision_score, recall_score, f1_score, accuracy_score

# NOTE: Geoprocessing libraries are kept but will only be useful if specific classes are vectorized.
import rasterio.features
from shapely.geometry import shape

# Import utilities from the other files
from data_utils_unet import SegmentationDataset
from model_utils_unet import get_model, combined_loss, iou_score

# --- Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 10
NUM_EPOCHS = 50 # Set to 50 as requested
LR = 1e-4
IMAGE_SIZE = 128
NUM_CLASSES = 7

# Class Names for visualization
CLASS_NAMES = [
    "Unlabelled", "Vegetation", "Built-Up", "Informal Settlements",
    "Impervious Surfaces", "Barren", "Water"
]

# --- Data Path Placeholders ---
BASE_PATH = "../../Dataset/Dataset/Prepared_Dataset"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train/images")
TRAIN_MASK_DIR = os.path.join(BASE_PATH, "train/masks")
VAL_IMG_DIR = os.path.join(BASE_PATH, "val/images")
VAL_MASK_DIR = os.path.join(BASE_PATH, "val/masks")


# --- Augmentations ---
train_transform = A.Compose([
    A.RandomRotate90(),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.Affine(rotate=(-15,15), scale=(0.9,1.1), translate_percent=(0.06,0.06)),
    A.RandomBrightnessContrast(p=0.5),
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

# --- Evaluation Function ---
def validate_epoch(model, val_loader):
    model.eval()
    val_loss_sum = 0.0
    val_miou_sum = 0.0
    all_preds = []
    all_masks = []
    
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(DEVICE)
            masks = masks.to(DEVICE).squeeze(-1).long()
            
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = combined_loss(logits, masks)
            
            val_loss_sum += loss.item() * imgs.size(0)
            preds = torch.argmax(logits, dim=1)
            val_miou_sum += iou_score(logits, masks, num_classes=NUM_CLASSES) * imgs.size(0)
            
            all_preds.append(preds.cpu().numpy())
            all_masks.append(masks.cpu().numpy())

    avg_loss = val_loss_sum / len(val_loader.dataset)
    avg_miou = val_miou_sum / len(val_loader.dataset)
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)
    
    return avg_loss, avg_miou, all_preds, all_masks

# --- Plotting Function ---
def plot_results(
    train_losses, val_losses, val_mious,
    all_val_preds, all_val_masks,
    val_dataset,
    session_title,
    num_sample_images=5
):
    print(f"\n--- Generating Visualizations for: {session_title} ---")
    
    plt.figure(figsize=(12, 5))
    plt.suptitle(session_title, fontsize=16)

    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(val_mious, label='Validation mIoU', color='orange')
    plt.title('Validation Mean IoU (mIoU)')
    plt.xlabel('Epoch')
    plt.ylabel('mIoU')
    plt.legend()
    plt.grid(True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show(block=False) # Show plot but don't block execution yet
    
    flat_true = all_val_masks.flatten()
    flat_pred = all_val_preds.flatten()
    
    cm = confusion_matrix(flat_true, flat_pred, labels=np.arange(NUM_CLASSES))
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix - {session_title}')
    plt.show(block=False)

    overall_accuracy = accuracy_score(flat_true, flat_pred)
    overall_miou_macro = jaccard_score(flat_true, flat_pred, average='macro', zero_division=0)
    f1_macro = f1_score(flat_true, flat_pred, average='macro', zero_division=0)
    
    print(f"\n--- Quantitative Metrics ({session_title}) ---")
    print(f"Overall Pixel Accuracy: {overall_accuracy:.4f}")
    print(f"Mean IoU (Jaccard Index) [Macro]: {overall_miou_macro:.4f}")
    print(f"F1-Score [Macro]: {f1_macro:.4f}")

    precision_per_class = precision_score(flat_true, flat_pred, average=None, labels=np.arange(NUM_CLASSES), zero_division=0)
    recall_per_class = recall_score(flat_true, flat_pred, average=None, labels=np.arange(NUM_CLASSES), zero_division=0)
    
    print("\n--- Per-Class Metrics ---")
    print(f"{'Class':<20} | {'Precision':<10} | {'Recall':<10}")
    print("-" * 45)
    for i in range(NUM_CLASSES):
        print(f"{CLASS_NAMES[i]:<20} | {precision_per_class[i]:<10.4f} | {recall_per_class[i]:<10.4f}")

    cmap = plt.cm.get_cmap('viridis', NUM_CLASSES)
    plt.figure(figsize=(15, num_sample_images * 4))
    sample_indices = np.random.choice(len(val_dataset), num_sample_images, replace=False)
    
    for i, idx in enumerate(sample_indices):
        original_img_tensor, true_mask_tensor = val_dataset[idx]
        
        img_display = original_img_tensor.permute(1, 2, 0).numpy()
        mean = np.array([0.485,0.456,0.406])
        std = np.array([0.229,0.224,0.225])
        img_display = std * img_display + mean
        img_display = np.clip(img_display, 0, 1)

        true_mask_display = true_mask_tensor.squeeze().numpy()
        # Find the correct prediction for the sampled index
        original_index_in_val_set = val_dataset.images.index(val_dataset.images[idx])
        predicted_mask_display = all_val_preds[original_index_in_val_set].squeeze()

        plt.subplot(num_sample_images, 3, i * 3 + 1)
        plt.imshow(img_display)
        plt.title('Original Image')
        plt.axis('off')

        plt.subplot(num_sample_images, 3, i * 3 + 2)
        plt.imshow(true_mask_display, cmap=cmap, vmin=0, vmax=NUM_CLASSES-1)
        plt.title('Ground Truth Mask')
        plt.axis('off')

        plt.subplot(num_sample_images, 3, i * 3 + 3)
        plt.imshow(predicted_mask_display, cmap=cmap, vmin=0, vmax=NUM_CLASSES-1)
        plt.title('Predicted Mask')
        plt.axis('off')
        
    plt.suptitle(f"Visual Examples - {session_title}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    # Use block=True on the last plot to pause execution until graphs are closed
    plt.show(block=True) 

# --- Training Session Function ---
def run_training_session(enhancement_mode, model_save_path):
    print("\n" + "="*50)
    print(f"STARTING TRAINING SESSION: Enhancement Mode = '{enhancement_mode}'")
    print(f"Model will be saved to: {model_save_path}")
    print("="*50 + "\n")

    # 1. DataLoaders
    train_dataset = SegmentationDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform, use_enhancement=enhancement_mode)
    # Validation dataset should not be enhanced to ensure fair comparison across models
    val_dataset = SegmentationDataset(VAL_IMG_DIR, VAL_MASK_DIR, transform=val_transform, use_enhancement='none')
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    # 2. Model, Optimizer, Scheduler
    model = get_model(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, mode='max')
    scaler = torch.cuda.amp.GradScaler()

    best_miou = 0.0
    train_losses_history, val_losses_history, val_mious_history = [], [], []
    final_val_preds, final_val_masks = None, None

    # 3. Training Loop
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} (Train)")
        for imgs, masks in progress_bar:
            imgs = imgs.to(DEVICE)
            masks = masks.to(DEVICE).squeeze(-1).long()
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = combined_loss(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * imgs.size(0)
            progress_bar.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader.dataset)
        train_losses_history.append(avg_train_loss)

        val_loss, val_miou, current_epoch_preds, current_epoch_masks = validate_epoch(model, val_loader)
        val_losses_history.append(val_loss)
        val_mious_history.append(val_miou)
        
        scheduler.step(val_miou)

        print(f"Epoch {epoch} finished. Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val mIoU: {val_miou:.4f}")

        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), model_save_path)
            print(f"Model saved to {model_save_path}!")
            final_val_preds = current_epoch_preds
            final_val_masks = current_epoch_masks

    # 4. Generate and Plot Results for the session
    if not os.path.exists(model_save_path):
        print(f"Warning: Best model was not saved. Using last epoch model state for plotting.")
    else:
        print(f"Loading best model from {model_save_path} for final evaluation.")
        model.load_state_dict(torch.load(model_save_path))
        _, _, final_val_preds, final_val_masks = validate_epoch(model, val_loader)

    plot_results(
        train_losses_history,
        val_losses_history,
        val_mious_history,
        final_val_preds,
        final_val_masks,
        val_dataset,
        session_title=f"U-Net++ | Enhancement: {enhancement_mode}"
    )

# --- Main Execution ---
def main():
    if not os.path.exists(TRAIN_IMG_DIR) or not os.path.exists(VAL_IMG_DIR):
        print(f"!! ERROR: Data paths are incorrect. Expected: {BASE_PATH}")
        return

    # --- Run Training Session 1: No Enhancement ---
    run_training_session(enhancement_mode='none', model_save_path="unet_plus_none_50.pth")

    # --- Run Training Session 2: With Preprocessing/Enhancement ---
    run_training_session(enhancement_mode='all', model_save_path="unet_plus_preprocess_50.pth")
    
    # --- Run Training Session 3: Hybrid Enhancement ---
    run_training_session(enhancement_mode='hybrid', model_save_path="unet_plus_hybrid_50.pth")

if __name__ == '__main__':
    try:
        main()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


STARTING TRAINING SESSION: Enhancement Mode = 'none'
Model will be saved to: unet_plus_none_50.pth



C:\Users\veerk\AppData\Local\Temp\ipykernel_5276\1502536598.py:215: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
c:\Users\veerk\OneDrive\Desktop\DIP Project\dip\Lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
Epoch 1/50 (Train):   0%|          | 0/713 [00:00<?, ?it/s]c:\Users\veerk\OneDrive\Desktop\DIP Project\dip\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch 1/50 (Train):   0%|          | 0/713 [00:17<?, ?it/s]


KeyboardInterrupt: 